# 0. Подготовка

In [ ]:
import os
# from pathlib import Path
# import shutil
import json
import requests
import re

# Константы
MYINDIE_JAM_URL = "https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page="
MYINDIE_JAM_URL_PAGES = 3  # Количество страниц с играми на геймджеме
OUTPUT_DIR = "output/"

# Директория для джема
jam_path = re.search(r'https://myindie.ru/jams/jam/([^<]+)/games', MYINDIE_JAM_URL)
if jam_path is None:
    raise ValueError("Не удалось извлечь путь джема из URL. Проверьте правильность MYINDIE_JAM_URL.")
jam_path = str(jam_path.group(1))
os.makedirs(os.path.join(OUTPUT_DIR, jam_path), exist_ok=True)

# 1. Скачиваем все страницы игр геймджема

1.1 Ищем все игры джема

In [3]:
def get_jam_games_urls(jam_url):
    """Скачивает страницу джема и извлекает URL игр."""

    response = requests.get(jam_url)

    if response.status_code != 200:
        print(f"Ошибка при получении страницы джема: `{response.status_code}`")
        return []

    games_urls = re.findall(r'/games/game/[\w-]+', response.text)
    games_urls = [f"https://myindie.ru{url}" for url in games_urls]
    print(f"Найдено {len(games_urls)} URL игр в: {jam_url}")

    return games_urls


games_urls = []
for page in range(1, MYINDIE_JAM_URL_PAGES + 1):
    paged_url = f"{MYINDIE_JAM_URL}{page}"
    games_urls.extend(get_jam_games_urls(paged_url))

print(f"\nНайдено {len(games_urls)} игр на {MYINDIE_JAM_URL_PAGES} страницах:\n{chr(10).join(games_urls)}")


Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=1
Найдено 30 URL игр в: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=2
Найдено 8 URL игр в: https://myindie.ru/jams/jam/myindie-game-jam-level-9/games?page=3

Найдено 68 игр на 3 страницах:
https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
https://myindie.ru/games/game/trash-under-ground
https://myindie.ru/games/game/cult-indie
https://myindie.ru/games/game/udivitelnyj-ded
https://myindie.ru/games/game/franshiza-ktulhu_nx3
https://myindie.ru/games/game/durdom
https://myindie.ru/games/game/exorcismo
https://myindie.ru/games/game/5-days-with-the-necronomicon_opw
https://myindie.ru/games/game/full-moon-twin-rite
https://myindie.ru/games/game/zayachij-kult
https://myindie.ru/games/game/no-edward
https://myindie.ru/games/game/protokol-pentagrammy
https://myindie.ru/games/game/all-hail-sister
https://myindie.ru/games/game/s-run_5pm
https://myindie.ru/games/game/les
https://my

1.2 Скачать все HTML-страницы игр геймджема

In [ ]:
jam_games_file_path = os.path.join(OUTPUT_DIR + jam_path, f"jam_games.txt")
if os.path.exists(jam_games_file_path):
    os.remove(jam_games_file_path)
jam_games_file = open(jam_games_file_path, 'a', encoding='utf-8')

# DEBUG_GAMES_MAX = 2  # DEBUG delete
for i, game_url in enumerate(games_urls): #[:DEBUG_GAMES_MAX]):
    print(f"Обработка URL игры: {game_url}")
    response = requests.get(game_url)
    if response.status_code != 200:
        print(f"* Ошибка при получении страницы игры: `{response.status_code}`")
        continue

    title_match = re.search(r'<title>([^<]+)</title>', response.text)
    if title_match:
        title = f"{i:03d}_" + re.sub(r'[^\w_]', '-', re.sub(r'\s+', '_', title_match.group(1).strip()))
    else:
        title = f"{i:03d}_" + "Unknown"

    output_file_path = os.path.join(OUTPUT_DIR + jam_path, f"{title}.html")
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write(response.text)
        print(f"Сохранено в `{output_file_path}`")
    jam_games_file.write(f"{output_file_path} {game_url}\n")

jam_games_file.close()

Обработка URL игры: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
Сохранено в `output/000_УЛЬТИМАТУМ--_Одна_ночь_с_Карачуном-_Жанр-_Horror_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/trash-under-ground
Сохранено в `output/001_Fetidity-_Жанр-_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/cult-indie
Сохранено в `output/002_Cult_Indie-_Жанр-_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/udivitelnyj-ded
Сохранено в `output/003_Удивительный_Дед-_Жанр-_Platformer_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/franshiza-ktulhu_nx3
Сохранено в `output/004_Франшиза_Ктулху_-_Жанр-_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/durdom
Сохранено в `output/005_Durdom-_Жанр-_Shooter_-_Инди-игры_-_MyIndie.html`
Обработка URL игры: https://myindie.ru/games/game/exorcismo
Сохранено в `output/006_Exorcismo-_Жанр-_Shooter-_Surviva

# 2. Извлекаем данные со всех страниц игр
2.1 Используем уже скачанные HTML-страницы игр геймджема, чтобы извлечь данные о каждой игре

In [22]:
def unflatten_nuxt_data(data):
    """ Разворачивает плоскую структуру данных Nuxt.js в дерево. """
    if not isinstance(data, list) or not data: return data
    memo = {}
    def resolve(val):
        if isinstance(val, int) and 0 <= val < len(data):
            if val not in memo:
                memo[val] = resolve_item(data[val])
            return memo[val]
        return val
    def resolve_item(item):
        if isinstance(item, dict):
            return {k: resolve(v) for k, v in item.items()}
        if isinstance(item, list):
            return [resolve(v) for v in item]
        return item
    return resolve_item(data[1])

jam_games_file_path = os.path.join(OUTPUT_DIR, "jam_games.txt")
jam_games_file = open(jam_games_file_path, 'r', encoding='utf-8')
print(f"Всего игр в файле `{jam_games_file_path}`: {len(jam_games_file.readlines())}")

Всего игр в файле `output/jam_games.txt`: 68


2.2 Собираем судейские оценки и отзывы для каждой игры

In [23]:
all_judges_reviews = []

jam_games_file.seek(0)
for line in jam_games_file:
    output_file_path, game_url = line.strip().split(' ', 1)
    print(f"Обработка: {game_url}")
    with open(output_file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    match = re.search(r'id=\"__NUXT_DATA__\">([^<]+)</script>', html_content)
    if not match:
        print(f"* Не найден __NUXT_DATA__ в `{output_file_path}`")
        continue

    json_data = json.loads(match.group(1))
    unflattened = unflatten_nuxt_data(json_data)
    if not unflattened:
        print(f"* Не удалось развернуть данные Nuxt.js в `{output_file_path}`")
        continue

    data_section = unflattened.get('data', [])
    if not data_section:
        print(f"* Не найден раздел 'data' в развернутых данных Nuxt.js в `{output_file_path}`")
        continue

    # Берем второй элемент списка (индекс 1) — там словарь с результатами
    if isinstance(data_section, list) and len(data_section) > 1:
        payload_container = data_section[1]
        if not payload_container:
            print(f"* Пустой контейнер в `{output_file_path}`")
            continue

        # В словаре берем первый ключ (game<alias>)
        if isinstance(payload_container, dict) and payload_container:
            first_key = list(payload_container.keys())[0]
            game_payload = payload_container[first_key]
            if not game_payload:
                print(f"* Пустой объект игры в `{output_file_path}`")
                continue

            # Извлекаем отзывы из game_payload['data']['reviews']
            if isinstance(game_payload, dict):
                inner_data = game_payload.get('data', {})
                if not inner_data:
                    print(f"* Пустой объект 'data' в `{output_file_path}`")
                    continue
                reviews = inner_data.get('reviews', [])
                if not reviews:
                    print(f"* Не найдено отзывов судей в `{output_file_path}`")
                    continue

                judges = [r for r in reviews if isinstance(r, dict) and r.get('reviewerType') == 'judge']
                all_judges_reviews.extend(judges)
                print(f"Всего судейских отзывов: {len(judges)} для игры `{game_url}`\n")


Обработка: https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom
Всего судейских отзывов: 3 для игры `https://myindie.ru/games/game/ultimatum-odna-noch-s-karachunom`

Обработка: https://myindie.ru/games/game/trash-under-ground
Всего судейских отзывов: 0 для игры `https://myindie.ru/games/game/trash-under-ground`

Обработка: https://myindie.ru/games/game/cult-indie
Всего судейских отзывов: 4 для игры `https://myindie.ru/games/game/cult-indie`

Обработка: https://myindie.ru/games/game/udivitelnyj-ded
Всего судейских отзывов: 1 для игры `https://myindie.ru/games/game/udivitelnyj-ded`

Обработка: https://myindie.ru/games/game/franshiza-ktulhu_nx3
Всего судейских отзывов: 2 для игры `https://myindie.ru/games/game/franshiza-ktulhu_nx3`

Обработка: https://myindie.ru/games/game/durdom
* Не найдено отзывов судей в `output/005_Durdom-_Жанр-_Shooter_-_Инди-игры_-_MyIndie.html`
Обработка: https://myindie.ru/games/game/exorcismo
Всего судейских отзывов: 0 для игры `https://myindie.ru/game

4. Завершение работы

In [28]:
jam_games_file.close()
print(json.dumps(all_judges_reviews, ensure_ascii=False, indent=2))

[
  {
    "id": "986411ed-f668-45d0-8b1c-e57160c42b55",
    "userId": "c739f699-9424-4039-9c0b-8b26a6f0f1c2",
    "user": {
      "id": "c739f699-9424-4039-9c0b-8b26a6f0f1c2",
      "username": "PoliKhai",
      "alias": "polikhai",
      "profile": {
        "id": "34884c81-3b86-4690-a673-fafc34111c16",
        "avatar": "/users/c739f699-9424-4039-9c0b-8b26a6f0f1c2/1775489204277.jpeg",
        "banner": "b_kfmtkcjvrmb9fs.png",
        "bannerUrl": "/users/c739f699-9424-4039-9c0b-8b26a6f0f1c2/b_kfmtkcjvrmb9fs.png",
        "firstName": "Полина",
        "lastName": "Хайдукова",
        "about": null,
        "links": []
      },
      "createdAt": ""
    },
    "reviewerType": "judge",
    "score": 2.5,
    "criterias": {
      "art": 3,
      "sound": 3,
      "theme": 1.5,
      "gameplay": 1.5,
      "narrative": 4,
      "overall_impression": 2
    },
    "reviewText": "<p>Прикольный бешеный стиль и забавная озвучка. Касаемо идеи вообще не понятно, при чем тут оккультизм. Деталей п